# Система рекомендацій відеоконтенту — аналіз та порівняння алгоритмів кластеризації

**Автор:** Романів Іван Вікторович  
**Мета:** Дослідження датасету YouTube Trending Videos, порівняння алгоритмів кластеризації (K-Means, DBSCAN, Ієрархічна) та візуалізація результатів.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack

RAW_DATA_PATH = '../data/raw/USvideos.csv'
PROCESSED_DIR = '../data/processed/'

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
print('Бібліотеки завантажено успішно.')

## 1. Дослідницький аналіз даних (EDA)

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH, on_bad_lines='skip')
print(f'Розмір датасету: {df_raw.shape[0]} рядків, {df_raw.shape[1]} колонок')
df_raw.head(3)

In [ ]:
print('Типи даних та кількість пропущених значень:')
missing = df_raw.isnull().sum()
dtype_info = df_raw.dtypes
info_df = pd.DataFrame({'Тип': dtype_info, 'Пропущено': missing, '% пропущено': (missing / len(df_raw) * 100).round(2)})
info_df[info_df['Пропущено'] > 0]

In [ ]:
# Статистика числових колонок
num_cols = ['views', 'likes', 'comment_count']
df_raw[num_cols] = df_raw[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
df_raw[num_cols].describe().round(0)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    data = np.log1p(df_raw[col].dropna())
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(f'log1p({col})', fontsize=12)
    ax.set_xlabel('log1p значення')
    ax.set_ylabel('Частота')
    ax.grid(True, alpha=0.3)
plt.suptitle('Розподіл числових ознак (логарифмічна шкала)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Топ-10 каналів за кількістю трендових відео
top_channels = df_raw['channel_title'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 5))
top_channels.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Топ-10 каналів за кількістю трендових відео', fontsize=13)
ax.set_xlabel('Кількість відео в трендах')
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Кореляційна матриця числових ознак
df_raw['log_views'] = np.log1p(df_raw['views'])
df_raw['log_likes'] = np.log1p(df_raw['likes'])
df_raw['log_comments'] = np.log1p(df_raw['comment_count'])

corr = df_raw[['log_views', 'log_likes', 'log_comments']].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
labels = ['log(views)', 'log(likes)', 'log(comments)']
ax.set_xticks(range(3)); ax.set_xticklabels(labels, rotation=30)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', color='black', fontsize=12)
ax.set_title('Кореляційна матриця числових ознак', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Підготовка даних (TF-IDF + масштабування)

In [ ]:
# Завантажуємо вже навчені артефакти (або обробляємо заново)
features_path = os.path.join(PROCESSED_DIR, 'features.joblib')

if os.path.exists(features_path):
    X = joblib.load(features_path)
    print(f'Завантажено готову матрицю ознак: {X.shape}')
else:
    print('Матрицю ознак не знайдено. Запустіть main.py або build_features.py спочатку.')

In [ ]:
print(f'Форма матриці ознак: {X.shape}')
print(f'Тип матриці: {type(X).__name__}')
print(f'Розрідженість: {1 - X.nnz / (X.shape[0] * X.shape[1]):.1%} нулів')

## 3. Метод ліктя (Elbow Method)

In [ ]:
elbow_img = os.path.join(PROCESSED_DIR, 'elbow_method.png')
if os.path.exists(elbow_img):
    from IPython.display import Image
    display(Image(elbow_img))
else:
    # Будуємо графік тут
    k_range = range(2, 21)
    inertias = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init='auto')
        km.fit(X)
        inertias.append(km.inertia_)
        print(f'  k={k}: inertia={km.inertia_:.0f}')

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=6)
    ax.set_xlabel('Кількість кластерів (k)')
    ax.set_ylabel('Інерція (SSE)')
    ax.set_title('Метод ліктя для вибору оптимального k')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Зниження розмірності (PCA)

In [ ]:
X_dense = X.toarray() if hasattr(X, 'toarray') else np.array(X)

# PCA до 2D для візуалізації
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_dense)
print(f'PCA 2D — пояснена дисперсія: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%')

# PCA до 50D для DBSCAN та ієрархічної кластеризації
pca_50 = PCA(n_components=50, random_state=42)
X_50d = pca_50.fit_transform(X_dense)
print(f'PCA 50D — пояснена дисперсія: {pca_50.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# Графік пояснюваної дисперсії по компонентах
pca_full = PCA(n_components=50, random_state=42).fit(X_dense)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, 51), cumvar * 100, 'steelblue', linewidth=2)
ax.fill_between(range(1, 51), cumvar * 100, alpha=0.15, color='steelblue')
ax.axhline(y=80, color='red', linestyle='--', label='80% дисперсії')
ax.set_xlabel('Кількість компонент PCA')
ax.set_ylabel('Накопичена пояснена дисперсія (%)')
ax.set_title('PCA — накопичена пояснена дисперсія')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Навчання та порівняння алгоритмів кластеризації

In [ ]:
K = 15  # оптимальне значення за методом ліктя
results = {}

In [ ]:
# --- K-MEANS ---
print('Навчання K-Means...')
kmeans = KMeans(n_clusters=K, random_state=42, n_init='auto')
km_labels = kmeans.fit_predict(X)

sil_km = silhouette_score(X, km_labels, sample_size=2000, random_state=42)
db_km  = davies_bouldin_score(X_dense, km_labels)
ch_km  = calinski_harabasz_score(X_dense, km_labels)

results['K-Means'] = {'Silhouette': sil_km, 'Davies-Bouldin': db_km, 'Calinski-Harabasz': ch_km, 'K': K}
print(f'  Silhouette={sil_km:.4f}  DB={db_km:.4f}  CH={ch_km:.2f}')

In [ ]:
# --- DBSCAN (після PCA-50, автоматичний підбір eps) ---
print('Навчання DBSCAN (після PCA-50)...')

MIN_SAMPLES = 5

# Автоматичний підбір eps через k-distance графік
nbrs = NearestNeighbors(n_neighbors=MIN_SAMPLES, metric='euclidean', n_jobs=-1).fit(X_50d)
distances, _ = nbrs.kneighbors(X_50d)
k_distances = np.sort(distances[:, -1])

diffs = np.diff(k_distances)
knee_idx = int(np.argmax(diffs))
auto_eps = float(k_distances[knee_idx])
auto_eps = round(max(auto_eps, 0.3), 4)
print(f'  Автоматично визначений eps = {auto_eps}')

# Візуалізація k-distance графіку
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_distances, color='steelblue', linewidth=1.2)
ax.axvline(x=knee_idx, color='red', linestyle='--', label=f'Лікоть (eps={auto_eps})')
ax.axhline(y=auto_eps, color='orange', linestyle=':', label=f'eps = {auto_eps}')
ax.set_title(f'k-Distance графік (k={MIN_SAMPLES}) — визначення eps для DBSCAN', fontsize=13)
ax.set_xlabel('Точки (відсортовано за відстанню)')
ax.set_ylabel(f'Відстань до {MIN_SAMPLES}-го сусіда')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

dbscan = DBSCAN(eps=auto_eps, min_samples=MIN_SAMPLES, metric='euclidean', n_jobs=-1)
db_labels = dbscan.fit_predict(X_50d)

n_clusters_db = len(set(db_labels) - {-1})
n_noise = int((db_labels == -1).sum())
print(f'  Знайдено кластерів: {n_clusters_db} | Шумових точок: {n_noise} ({n_noise/len(db_labels)*100:.1f}%)')

# Резервний механізм: якщо кластерів <= 1 — зменшуємо eps
if n_clusters_db <= 1:
    print('  Автопідбір не дав результату, запускаємо резервні значення eps...')
    for fallback_eps in [auto_eps * 0.5, auto_eps * 0.3, 0.5, 0.3]:
        fallback_eps = round(fallback_eps, 4)
        print(f'    Спроба eps={fallback_eps}...')
        dbscan = DBSCAN(eps=fallback_eps, min_samples=MIN_SAMPLES, metric='euclidean', n_jobs=-1)
        db_labels = dbscan.fit_predict(X_50d)
        n_clusters_db = len(set(db_labels) - {-1})
        n_noise = int((db_labels == -1).sum())
        auto_eps = fallback_eps
        print(f'    → кластерів: {n_clusters_db}, шуму: {n_noise}')
        if n_clusters_db > 1:
            break

print(f'  Фінальний eps={auto_eps} | Кластерів: {n_clusters_db} | Шум: {n_noise}')

if n_clusters_db > 1:
    mask = db_labels != -1
    sil_db = silhouette_score(X_50d[mask], db_labels[mask],
                              sample_size=min(2000, mask.sum()), random_state=42)
    db_db  = davies_bouldin_score(X_50d[mask], db_labels[mask])
    ch_db  = calinski_harabasz_score(X_50d[mask], db_labels[mask])
    results['DBSCAN'] = {'Silhouette': sil_db, 'Davies-Bouldin': db_db,
                         'Calinski-Harabasz': ch_db, 'K': n_clusters_db}
    print(f'  Silhouette={sil_db:.4f}  DB={db_db:.4f}  CH={ch_db:.2f}')
else:
    results['DBSCAN'] = {'Silhouette': None, 'Davies-Bouldin': None,
                         'Calinski-Harabasz': None, 'K': n_clusters_db}
    print('  DBSCAN не виділив достатньо кластерів — метрики недоступні.')

In [ ]:
# --- ІЄРАРХІЧНА КЛАСТЕРИЗАЦІЯ (Ward) ---
print('Навчання AgglomerativeClustering (Ward, підвибірка 3000)...')
sample_idx = np.random.RandomState(42).choice(X_50d.shape[0], min(3000, X_50d.shape[0]), replace=False)
X_hier = X_50d[sample_idx]

agglo = AgglomerativeClustering(n_clusters=K, linkage='ward')
hier_labels = agglo.fit_predict(X_hier)

sil_hier = silhouette_score(X_hier, hier_labels, sample_size=min(2000, len(X_hier)), random_state=42)
db_hier  = davies_bouldin_score(X_hier, hier_labels)
ch_hier  = calinski_harabasz_score(X_hier, hier_labels)

results['Hierarchical (Ward)'] = {'Silhouette': sil_hier, 'Davies-Bouldin': db_hier, 'Calinski-Harabasz': ch_hier, 'K': K}
print(f'  Silhouette={sil_hier:.4f}  DB={db_hier:.4f}  CH={ch_hier:.2f}')

In [ ]:
# --- ПОРІВНЯЛЬНА ТАБЛИЦЯ ---
comparison_df = pd.DataFrame(results).T
comparison_df.index.name = 'Алгоритм'
comparison_df.columns = ['Silhouette Score ↑', 'Davies-Bouldin ↓', 'Calinski-Harabasz ↑', 'Кластерів']
comparison_df

In [ ]:
# Візуалізація порівняльних метрик
metrics = ['Silhouette Score ↑', 'Davies-Bouldin ↓']
algos = [a for a in comparison_df.index if comparison_df.loc[a, 'Silhouette Score ↑'] is not None]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, metric, color_list in zip(axes, metrics, [colors, colors]):
    vals = [comparison_df.loc[a, metric] for a in algos]
    bars = ax.bar(algos, vals, color=colors[:len(algos)], edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=12)
    ax.set_ylabel('Значення метрики')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.4f}',
                ha='center', va='bottom', fontsize=10)
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Порівняння алгоритмів кластеризації за метриками', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Візуалізація кластерів (PCA 2D)

In [ ]:
# Беремо підвибірку для швидкої візуалізації
sample_size_vis = min(3000, X_2d.shape[0])
vis_idx = np.random.RandomState(42).choice(X_2d.shape[0], sample_size_vis, replace=False)
X_vis = X_2d[vis_idx]
km_labels_vis = km_labels[vis_idx]

fig, ax = plt.subplots(figsize=(12, 8))
cmap = cm.get_cmap('tab20', K)

for cluster_id in range(K):
    mask = km_labels_vis == cluster_id
    ax.scatter(X_vis[mask, 0], X_vis[mask, 1],
               c=[cmap(cluster_id)], label=f'Кластер {cluster_id}',
               alpha=0.6, s=20, edgecolors='none')

ax.set_title(f'K-Means кластеризація (k={K}), PCA 2D проєкція', fontsize=14)
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% дисперсії)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% дисперсії)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8, ncol=2)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Розмір кластерів K-Means
cluster_sizes = pd.Series(km_labels).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(12, 4))
cluster_sizes.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Розмір кластерів (K-Means)', fontsize=13)
ax.set_xlabel('Номер кластера')
ax.set_ylabel('Кількість відео')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Розподіл по кластерах:')
print(cluster_sizes.to_string())

## 7. Висновки

| Критерій | K-Means | DBSCAN | Ієрархічна (Ward) |
|---|---|---|---|
| Масштабованість | Висока | Середня | Низька |
| Чутливість до шуму | Низька | Висока | Низька |
| Задає k наперед | Так | Ні | Так |
| Придатність для TF-IDF | Висока | Потребує PCA | Потребує PCA |

**Обраний алгоритм:** K-Means з k=15 — найкращий баланс між якістю кластеризації та швидкодією для даного датасету.